## Experimento de Escalado: Extracción de 1.000 Prospectos (Fase Descartada)

**Contexto del Experimento:** En esta fase del proyecto, se planteó la hipótesis de que aumentar el tamaño del corpus a $N=1.000$ prospectos permitiría una mayor finura en la detección de tópicos minoritarios. 

Los objetivos de este notebook son realizar una minería de datos extensiva sobre la API de CIMA para duplicar el volumen del corpus previo y evaluar si la densidad de datos adicional mejora la cohesión de los 12 clústeres o si, por el contrario, introduce ruido estadístico.

Este enfoque fue finalmente **descartado** en favor de la muestra de 500 prospectos altamente curada y deduplicada. El análisis de este cuaderno demostró que superar el umbral de los 500 documentos incurre en un "Rendimiento Marginal Decreciente": no se aporta diversidad semántica nueva, sino que se refuerza la redundancia de las plantillas legales de la AEMPS, dificultando la labor de abstracción de los LLMs.

## Extracción de prospectos

In [ ]:
import requests
import time
import pandas as pd

def extraer_datos_cima_unicos(limit=200):
    base_url = "https://cima.aemps.es/cima/rest"
    headers = {"Accept": "application/json"}
    prospectos_lista = []
    medicamentos_vistos = set() # Aquí guardamos la "primera palabra" del nombre
    
    pagina = 1
    total_capturados = 0

    while total_capturados < limit:
        params_busqueda = {'pagina': pagina, 'prospecto': 1}
        r_listado = requests.get(f"{base_url}/medicamentos", params=params_busqueda)
        
        if r_listado.status_code != 200: break
        medicamentos = r_listado.json().get('resultados', [])
        
        if not medicamentos: break # No hay más resultados

        for med in medicamentos:
            if total_capturados >= limit: break
            
            # 1. Normalizamos el nombre para el check
            # Tomamos la primera palabra
            nombre_raiz = med['nombre'].split()[0].upper()
            
            # 2. Check de unicidad
            if nombre_raiz in medicamentos_vistos:
                continue # Saltamos al siguiente medicamento de la lista
            
            nreg = med['nregistro']
            url_doc = f"{base_url}/docSegmentado/contenido/2"
            
            try:
                r_doc = requests.get(url_doc, params={'nregistro': nreg}, headers=headers)
                
                if r_doc.status_code == 200:
                    secciones = r_doc.json() 
                    if not secciones: continue
                    
                    for sec in secciones:
                        prospectos_lista.append({
                            'nregistro': nreg,
                            'nombre_med': med['nombre'],
                            'nombre_raiz': nombre_raiz,
                            'id_seccion_original': sec.get('seccion'),
                            'titulo_original': sec.get('titulo'),
                            'texto': sec.get('contenido')
                        })
                    
                    medicamentos_vistos.add(nombre_raiz)
                    total_capturados += 1
                    print(f"[{total_capturados}/{limit}] Nuevo: {nombre_raiz} ({med['nombre']})")
                
            except Exception as e:
                print(f"Error en {nreg}: {e}")
            
            time.sleep(0.1)
        
        pagina += 1
    
    return pd.DataFrame(prospectos_lista)

df_prospectos = extraer_datos_cima_unicos(1000)

[1/1000] Nuevo: A.A.S. (A.A.S. 100 mg COMPRIMIDOS)
[2/1000] Nuevo: ABACAVIR/LAMIVUDINA (ABACAVIR/LAMIVUDINA DR. REDDYS 600 MG/300 MG COMPRIMIDOS RECUBIERTOS CON PELICULA EFG)
[3/1000] Nuevo: ABIRATERONA (ABIRATERONA GLENMARK 500 MG COMPRIMIDOS RECUBIERTOS CON PELICULA EFG)
[4/1000] Nuevo: ABRAXANE (ABRAXANE 5 MG/ML POLVO PARA DISPERSION PARA PERFUSION)
[5/1000] Nuevo: ABRILIA (ABRILIA 20 MG COMPRIMIDOS EFG)
[6/1000] Nuevo: ABRYSVO (ABRYSVO POLVO Y DISOLVENTE PARA SOLUCION INYECTABLE)
[7/1000] Nuevo: ABSORCOL (ABSORCOL 10 mg COMPRIMIDOS)
[8/1000] Nuevo: ACABEL (ACABEL RAPID 8 mg COMPRIMIDOS RECUBIERTOS CON PELICULA)
[9/1000] Nuevo: ACALKA (ACALKA 1080 mg COMPRIMIDOS DE LIBERACION PROLONGADA)
[10/1000] Nuevo: ACARBOSA (ACARBOSA TECNIGEN 100 mg COMPRIMIDOS)
[11/1000] Nuevo: ACECLOFENACO (ACECLOFENACO CINFA 100 mg COMPRIMIDOS RECUBIERTOS CON PELICULA EFG)
[12/1000] Nuevo: ACEOTO (ACEOTO PLUS 3 mg/ml + 0,25 mg/ml GOTAS ÓTICAS EN SOLUCIÓN)
[13/1000] Nuevo: ACETATO (ACETATO DE CIPROTERONA/ETI

In [ ]:
from bs4 import BeautifulSoup
import re
import pandas as pd

from bs4 import BeautifulSoup
import re
import pandas as pd

def segmentar_con_contexto(df):
    filas_parrafos = []
    
    for _, row in df.iterrows():
        html = row['texto']
        if not html or pd.isna(html): continue
        
        soup = BeautifulSoup(html, "html.parser")
        
      
        for script in soup(["script", "style"]):
            script.decompose()

        # Iteramos por los hijos directos del body o el div principal
        elementos = soup.find_all(['p', 'ul', 'ol', 'h3', 'h4'])
        
        i = 0
        while i < len(elementos):
            el = elementos[i]
            texto_bloque = el.get_text(separator=' ', strip=True)
            
            if (texto_bloque.endswith(':') or "si se encuentra" in texto_bloque.lower()) and (i + 1 < len(elementos)):
                siguiente_el = elementos[i+1]
                texto_siguiente = siguiente_el.get_text(separator=' ', strip=True)
                
                # Unimos encabezado + lista
                texto_final = f"{texto_bloque} {texto_siguiente}"
                i += 2 
            else:
                texto_final = texto_bloque
                i += 1
            
    
            texto_final = re.sub(r'\s+', ' ', texto_final).strip()
            
            # Filtro de calidad (evitar ruidos muy cortos)
            if len(texto_final) > 50:
                filas_parrafos.append({
                    'nregistro': row['nregistro'],
                    'nombre_med': row['nombre_med'],
                    'titulo_original': row['titulo_original'],
                    'parrafo_limpio': texto_final
                })
                
    return pd.DataFrame(filas_parrafos)

# Aplicar el nuevo segmentador
df_parrafos = segmentar_con_contexto(df_prospectos)

# Eliminar duplicados exactos
df_parrafos = df_parrafos.drop_duplicates(subset=['parrafo_limpio'])

In [ ]:
import re

def preparar_lista_limpieza(df):
    nombres_sucios = df['nombre_med'].unique().tolist()
    lista_final = set()

    for nombre in nombres_sucios:

        lista_final.add(nombre.strip())
        
    
        marca = nombre.split()[0]
        if len(marca) > 3: 
            lista_final.add(marca)
            
        # 3. Extraemos todo lo que haya antes de la primera cifra (dosis)
        antes_dosis = re.split(r'\d', nombre)[0].strip()
        if len(antes_dosis) > 3:
            lista_final.add(antes_dosis)

    # Ordenamos de más largo a más corto para que el regex no rompa palabras
    return sorted(list(lista_final), key=len, reverse=True)

def anonimizar_texto(texto, lista_limpieza):
    if not texto: return ""
    
    # Usamos una sola pasada de regex para todas las marcas 
    # Escapamos los nombres por si tienen puntos o paréntesis como "A.A.S."
    patron = r'\b(' + '|'.join(map(re.escape, lista_limpieza)) + r')\b'
    
    # Reemplazamos ignorando mayúsculas/minúsculas
    return re.sub(patron, "[MEDICAMENTO]", texto, flags=re.IGNORECASE)



lista_negra = preparar_lista_limpieza(df_parrafos)
lista_negra += ["A.A.S. 100 mg"]

# Aplicamos la limpieza a la columna de párrafos
df_parrafos['parrafo_anonimizado'] = df_parrafos['parrafo_limpio'].apply(
    lambda x: anonimizar_texto(x, lista_negra)
)


In [30]:
df_parrafos.drop_duplicates(subset=["parrafo_anonimizado"], inplace=True)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

def eliminar_duplicados_semanticos(df, columna='parrafo_anonimizado', umbral=0.95):
    """
    Identifica y elimina párrafos con una similitud superior al umbral (95%).
    """
    print(f"Iniciando limpieza de {len(df)} filas...")
    
    # 1. Vectorizar
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df[columna].astype(str))
    
    # 2. Usar NearestNeighbors
    # Buscamos los 2 más cercanos
    nn = NearestNeighbors(n_neighbors=2, metric='cosine', n_jobs=-1)
    nn.fit(tfidf_matrix)
    
    distances, indices = nn.kneighbors(tfidf_matrix)
    
    # 3. Filtrar
    a_eliminar = set()
    for i in range(len(df)):
        if i in a_eliminar:
            continue
        
        # distance = 1 - similitud_coseno. Si sim > 0.95, dist < 0.05
        distancia = distances[i, 1]
        indice_vecino = indices[i, 1]
        
        if distancia < (1 - umbral):
            a_eliminar.add(indice_vecino)
            
    df_limpio = df.drop(df.index[list(a_eliminar)])
    print(f"Filas eliminadas: {len(a_eliminar)}")
    return df_limpio

df_parrafos = eliminar_duplicados_semanticos(df_parrafos, umbral=0.95)
df_parrafos.to_csv('prospectos_unicos_semanticos_v2.csv', index=False)

Iniciando limpieza de 59289 filas...
Filas eliminadas: 4747


## Cálculo de embeddings y clustering

In [32]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Cargar modelo de embeddings (multilingüe para español)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 1. Limpiar y vectorizar
textos = df_parrafos['parrafo_anonimizado'].tolist()
embeddings = model.encode(textos, show_progress_bar=True)

# 2. Clustering
num_clusters = 12 
clustering_model = KMeans(n_clusters=num_clusters, random_state=42)
df_parrafos['cluster'] = clustering_model.fit_predict(embeddings)

c:\Users\04jul\Desktop\CDIA\TFG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 432.45it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1705/1705 [29:11<00:00,  1.03s/it]


In [33]:
df_parrafos.to_csv("prospectos_clusters12_ampliado.csv", index=False)

###  Arquitectura del Prompt: Abstracción Semántica y Role-Play
Para transformar los clústeres de texto técnico en preguntas útiles para el paciente, se define la función `generar_prompt_final`. Esta función aplica tres capas de ingeniería de instrucciones (*Prompt Engineering*):

1. **Role-Play Cognitivo:** Se fuerza al modelo a adoptar la identidad de un "Paciente con una duda urgente". Esto cambia el tono de la respuesta de un resumen enciclopédico a una duda de lenguaje natural.
2. **Restricciones Negativas Estrictas:** Se prohíben términos como "fragmento", "información" o "sección" para evitar que el LLM hable sobre el dataset en lugar de sobre el contenido médico. Asimismo, se prohíbe el uso de nombres propios de medicamentos para garantizar la **Universalidad** de la pregunta.
3. **Lógica de Filtrado de Relevancia (70-80%):** Se instruye al modelo para que ignore detalles que aparecen de forma aislada (ruido) y se centre en el núcleo temático que une a la mayoría de los párrafos del clúster.

**Propósito:** Lograr que, independientemente de si el clúster contiene párrafos sobre colirios o comprimidos, la pregunta resultante sea una "Categoría Universal" (ej: "¿Cómo debo aplicar este medicamento correctamente?").

In [34]:
import pandas as pd
df_parrafos = pd.read_csv("prospectos_clusters12_ampliado.csv")

In [ ]:
def generar_prompt_final(cluster_id, ejemplos):
  ejemplos_str = "\n- ".join(ejemplos)
      

  prompt = f"""[INST] <<SYS>>
  Eres un PACIENTE que tiene una duda urgente y busca la respuesta en estos fragmentos. 
  TU REGLA MÁXIMA: Escribe ÚNICAMENTE la pregunta que harías a tu médico.
  PROHIBIDO usar palabras como: 'texto', 'fragmento', 'prospecto', 'diseño', 'JSON', 'información', 'sección', 'analizar'.
  PROHIBIDO mencionar nombres propios de medicamentos.
  <</SYS>>
  Sigue este razonamiento interno, pero sin escribirlo en la salida:
  1. Este fragmento "Puede disminuir la capacidad para conducir o manejar maquinaria por contener propilenglicol, ya que puede producir síntomas parecidos a los del alcohol." responde a la pregunta "¿Puedo conducir después de tomar [MEDICAMENTO]?".
  2. Este fragmento "Estas dosis se pueden repetir cada 4 horas."  responde a "¿Cada cuánto tiempo me tengo que tomar [MEDICAMENTO]?".
  3. Este fragmento Los demás componentes son celulosa microcristalina (E460), carboximetilalmidón sódico (Tipo A) de patata, povidona K90 (E1201)" responde a "¿Cuál es la composición de este medicamento?"
  4. Teniendo esto en cuenta, ¿qué PREGUNTA UNIVERSAL Y DIRECTA responden todos los fragmentos del grupo?
  Basándote en estos fragmentos:
  {ejemplos_str}

  Identifica el tema que se repite en el 70-80% de los fragmentos y descarta los detalles que solo aparecen en uno solo y genera la PREGUNTA DIRECTA, GENERAL Y NO ESPECÍFICA que un paciente normal (no experto) haría. 

  Responde exclusivamente en este formato JSON:
  {{
    "pregunta_del_paciente": "Escribe aquí la pregunta"
  }}
  [/INST]"""
  return prompt

## Conexión con Ollama

In [36]:
import requests
import json
import pandas as pd

def consultar_ollama_stream(prompt, modelo_id):
    url = "https://wiig.dia.fi.upm.es/ollama/v1/chat/completions"
    payload = {
        "model": modelo_id,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True
    }
    
    texto_acumulado = ""
    try:
        with requests.post(url, json=payload, stream=True, timeout=800) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line: continue
                decoded = line.decode("utf-8").replace("data: ", "").strip()
                if decoded == "[DONE]": break
                try:
                    chunk = json.loads(decoded)
                    content = chunk["choices"][0].get("delta", {}).get("content", "")
                    texto_acumulado += content
                except: continue
        return texto_acumulado
    except Exception as e:
        return f"ERROR: {str(e)}"

### 4.2. Despliegue del Comité Definitivo y Generación (K=12)
Tras los problemas de infraestructura (Timeouts) y colapso multilingüe detectados con ciertos modelos en fases previas, se reconfigura el comité de inferencia para garantizar la máxima estabilidad y calidad semántica, ya que esta vez tratamos con más frases para los prompts. 

El comité definitivo está compuesto por:
1. **Qwen2.5 (32B):** Sustituye a DeepSeek-R1. Ofrece un rendimiento *State-of-the-Art* en seguimiento de instrucciones y abstracción sin los problemas de latencia y sobrecarga de red observados previamente.
2. **Gemma-3 (27B):** El modelo que demostró un rendimiento perfecto en la evaluación de la "Regla de Oro" y la adopción del rol de paciente.
3. **Llama3_hard (8B):** Se mantiene como línea base estricta (baja temperatura) para triangular los resultados.

En esta ejecución final, se elimina el truncamiento de párrafos, inyectando el contexto completo del clúster en la ventana de atención de los LLMs.

In [37]:
import requests

def listar_modelos_disponibles():
    url_base = "https://wiig.dia.fi.upm.es/ollama/api/tags"
    try:
        response = requests.get(url_base, timeout=10)
        if response.status_code == 200:
            modelos = [m['name'] for m in response.json().get('models', [])]
            print(f" Modelos disponibles: {modelos}")
            return modelos
        else:
            print(f" No se pudo listar modelos. Status: {response.status_code}")
    except Exception as e:
        print(f" Error de conexión: {e}")
    return []

modelos_reales = listar_modelos_disponibles()

 Modelos disponibles: ['deepseek-r1:7b', 'llama3_evaluator:latest', 'gpt-oss:120b', 'deepseek-r1:70b', 'gemma3:27b', 'gemma3:12b', 'gemma3:4b', 'gemma3:1b', 'mistral:7b', 'qwen3:30b', 'qwen3:14b', 'llama3.1:70b', 'llama3.1:8b', 'deepseek-r1:32b', 'nemotron-3-nano:30b', 'deepseek-v3.2:cloud', 'qwen2.5:32b', 'qwen3:1.7b', 'qwen3:8b', 'llama3_easy:latest', 'llama3_medium:latest', 'llama3_hard:latest', 'llama3_hard_Open:latest', 'llama3_medium_Open:latest', 'llama3_easy_Open:latest', 'llama3:latest']


In [38]:
anotadores = ["qwen2.5:32b", "llama3_hard:latest", "gemma3:27b"]
resultados = []

### 4.3. Inferencia por Lotes (Batch Processing)
El sistema itera sobre los 12 clústeres óptimos. Para cada espacio vectorial, extrae los párrafos representativos (la "variedad" del clúster) y los inyecta en el *Prompt* de Abstracción Semántica. Se implementan pausas de ejecución (`time.sleep`) para respetar los límites de la API local y garantizar que las respuestas de los modelos de más de 27 billones de parámetros se procesen sin pérdida de *tokens*.

In [39]:
import time

for c_id in sorted(df_parrafos['cluster'].unique()):
    print(f" Procesando Cluster {c_id}...")
    
    # 1. Selección de variedad
    df_c = df_parrafos[df_parrafos['cluster'] == c_id]
    ejemplos_variados = df_c["parrafo_anonimizado"].to_list()
    
    # 2. Generación del prompt
    prompt_texto = generar_prompt_final(c_id, ejemplos_variados)
    
    fila = {"cluster": c_id, "ejemplos_usados": "|".join(ejemplos_variados)}
    
    # 3. Consulta a los LLMs
    for modelo in anotadores:
        print(f"  -> Consultando {modelo}...")
        respuesta = consultar_ollama_stream(prompt_texto, modelo)
        print(respuesta)
        fila[f"res_{modelo.replace(':', '_')}"] = respuesta
        time.sleep(1) # Pausa para no saturar el servidor

    resultados.append(fila)

df_final = pd.DataFrame(resultados)
df_final.to_csv("preguntas_12topicos_originales.csv", index=False)

 Procesando Cluster 0...
  -> Consultando qwen2.5:32b...
{
  "pregunta_del_paciente": "¿Qué debo hacer si olvido tomar una dosis del medicamento?"
}
  -> Consultando llama3_hard:latest...
{
    "pregunta_del_paciente": "¿Cómo debo administrar [MEDICAMENTO]? ¿Cuál es el plazo máximo para usarlo después de la apertura?"
}
  -> Consultando gemma3:27b...
```json
{
  "pregunta_del_paciente": "¿Cómo debo tomar este medicamento y qué debo tener en cuenta durante el tratamiento?"
}
```
 Procesando Cluster 1...
  -> Consultando qwen2.5:32b...
{
  "pregunta_del_paciente": "¿Qué efectos adversos puede tener este medicamento y cómo afecta a la sangre o a los análisis de sangre?"
}
  -> Consultando llama3_hard:latest...
{
"pregunta_del_paciente": "¿Qué son los riesgos y posibles efectos secundarios del medicamento [MEDICAMENTO]? ¡Lo dejo todo en sus manos!"
}
  -> Consultando gemma3:27b...
```json
{
  "pregunta_del_paciente": "¿Cuáles son los riesgos y efectos secundarios que debo conocer al tomar 

In [ ]:
import pandas as pd
import json
import re

def extraer_pregunta_segura(texto):
    """
    Limpia y extrae la pregunta del JSON generado por el LLM, 
    gestionando texto basura y fallos de formato.
    """
    if pd.isna(texto) or "ERROR" in str(texto):
        return "Error en respuesta"

    match = re.search(r'\{.*\}', str(texto), re.DOTALL)
    
    if match:
        json_str = match.group()
        try:
            # Reemplazar comillas dobles repetidas 
            json_str = json_str.replace('""', '"')
            datos = json.loads(json_str)
            
            pregunta = datos.get('pregunta_final')
            
            if pregunta:
                return pregunta.strip()
        except Exception:
            pass

    # Si falla el JSON, buscamos patrones de texto 
    # Busca frases que empiecen por "¿" y terminen por "?"
    preguntas_encontradas = re.findall(r'¿[^?]+\?', str(texto))
    if preguntas_encontradas:
        # Devolvemos la última pregunta encontrada
        return preguntas_encontradas[-1].strip()

    return "No se pudo identificar la pregunta"

df_resultados = pd.read_csv('preguntas_12topicos_originales.csv')

anotadores = [col for col in df_resultados.columns if col.startswith('res_')]

for col in anotadores:
    # Creamos una columna limpia para cada modelo
    nombre_limpio = col.replace('res_', 'pregunta_')
    df_resultados[nombre_limpio] = df_resultados[col].apply(extraer_pregunta_segura)


columnas_formulario = ['cluster'] + [col for col in df_resultados.columns if col.startswith('pregunta_')]
print(df_resultados[columnas_formulario].head())

df_resultados[columnas_formulario].to_csv('preguntas_12_v4.csv', index=False)

   cluster                               pregunta_qwen2.5_32b  \
0        0  ¿Qué debo hacer si olvido tomar una dosis del ...   
1        1  ¿Qué efectos adversos puede tener este medicam...   
2        2  ¿Qué efectos adversos puedo experimentar si to...   
3        3                 No se pudo identificar la pregunta   
4        4  ¿Cuáles son los efectos secundarios más comune...   

                         pregunta_llama3_hard_latest  \
0  ¿Cuál es el plazo máximo para usarlo después d...   
1  ¿Qué son los riesgos y posibles efectos secund...   
2  ¿Qué debo hacer si tomo demasiado Micralax y m...   
3  ¿Qué es [MEDICAMENTO] y qué función tiene en e...   
4  ¿Qué efectos secundarios graves pueden tener l...   

                                 pregunta_gemma3_27b  
0  ¿Cómo debo tomar este medicamento y qué debo t...  
1  ¿Cuáles son los riesgos y efectos secundarios ...  
2  ¿Cuáles son los posibles efectos secundarios o...  
3  ¿Qué necesito saber sobre los efectos secundar...

###  Auditoría del Comité (N=1.000): Colapso Semántico y Redundancia
La evaluación de las preguntas generadas por el comité avanzado (Qwen2.5, Llama3_hard, Gemma-3) sobre el corpus masivo de 1.000 prospectos revela un colapso estructural de la arquitectura de la información. 

En lugar de descubrir nuevos tópicos o afinar los existentes, el exceso de datos ha destruido la cohesión semántica lograda en iteraciones anteriores. Se identifican tres fallos críticos que invalidan este dataset:

##### 1. Fragmentación Masiva 
Aumentar la muestra no aportó diversidad, sino que multiplicó el texto legal estándar. Como resultado, **5 de los 12 clústeres (1, 2, 4, 10 y 11)** han colapsado en la misma temática: los efectos secundarios. Para un paciente, tener un prospecto con cinco secciones distintas que responden a "¿Qué efectos adversos puedo experimentar?" es funcionalmente inútil y cognitivamente agotador.

##### 2. Ruptura de las Restricciones Negativas (Sesgo de Especificidad)
A pesar de que el *System Prompt* prohibía explícitamente mencionar nombres de medicamentos o formatos, el ruido masivo de 1.000 documentos superó la capacidad de atención de los modelos. Llama3_hard vuelve a generar preguntas inválidas para un índice universal, preguntando por un fármaco específico (*"¿Qué debo hacer si tomo demasiado Micralax...?"* en el Cluster 2) y un formato físico (*"...usar la pluma precargada"* en el Cluster 7).

##### 3. Degradación del Role-Play Cognitivo
La saturación de lenguaje burocrático-legal desestabilizó el rol de "Paciente". En el Cluster 1, Llama3_hard abandona el tono neutro solicitado y genera una alucinación dramática (*"¡Lo dejo todo en sus manos!"*), evidenciando que el contexto sobrepasó los límites del *Prompt Engineering*.

### Conclusión Metodológica: La Falacia del Big Data Médico
Este experimento demuestra empíricamente el fenómeno de **Rendimiento Marginal Decreciente** en el Procesamiento de Lenguaje Natural aplicado a textos legales estructurados. 

La hipótesis inicial postulaba que $N=1.000$ prospectos generaría una taxonomía más rica. La realidad métrica demuestra lo contrario: no se añaden nuevas intenciones comunicativas, sino variaciones léxicas de la misma normativa de la AEMPS (saturación semántica). Esto fuerza al algoritmo de *Clustering* a sobreajustar (partir temas únicos en múltiples clústeres idénticos) y empuja a los LLMs a la alucinación.

**Decisión Final:**
Queda justificado el descarte de la aproximación de fuerza bruta masiva. Se confirma que **la calidad y pureza de los datos (Muestreo Estratégico de 500 prospectos + Deduplicación TF-IDF al 0.95)** es matemáticamente y clínicamente superior al volumen bruto. Esta validación cierra la fase de extracción y consolida la arquitectura de 12 Preguntas Maestras obtenida en el corpus reducido como el "Gold Standard" definitivo del proyecto.